### Importing Necessary Libraries



In [68]:
import os

from sklearn.linear_model import LogisticRegression

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE' #just for a glitch

import spacy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import string

In [3]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------------------- -------------- 8.1/12.8 MB 50.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 44.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Loading SpaCy's small english model

To get more details regarding SpaCy models check here : https://spacy.io/usage/models

In [4]:
# Loading Spacy small model as nlp
nlp = spacy.load("en_core_web_sm")

## Gathering all the Stop words which does not convey much meaning in the Sentiment

In [5]:
# Gathering all the stopwords
from spacy.lang.en.stop_words import STOP_WORDS
stopwords = list(STOP_WORDS)
print(len(stopwords)) #only 326 stopwords in spacy english small model

326


In [8]:
# Loading yelp dataset
data = pd.read_csv(r'dataFolder/train.csv', encoding="latin-1")
data = data[["text", "sentiment"]]
data = data.fillna("") #best and most efficient way (fillna) to get rid of null/NaN values
data.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


## So here we can deduce that Sentiment 1 is Positive and 0 is negative

In [11]:
data["sentiment"] = data["sentiment"].map({"negative": 0, "positive": 1, "neutral": 2}) #fastest way to replace items in panda DF using dict

In [9]:
print(data.shape)

(27481, 2)


In [15]:
# Sentiment ditribution in the dataset
data.sentiment.value_counts()

sentiment
2    11118
1     8582
0     7781
Name: count, dtype: int64

In [16]:
# Getting information regarding the null entries in the dataset
data.isnull().sum()

text         0
sentiment    0
dtype: int64

In [17]:
punct = string.punctuation
print(punct)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~




```
Here in the reviews we will find many stop words which do not add any meaning to the review.
Also punctuations will be encountered in the review which which will be considered as a seperate token by our model
So removing all the stop words and punctuation so that our model can train efficiently
```



In [30]:
doc = nlp("Today we are having heavy rainfall, We recommend you to stay at your home and be safe, Do not start running here and there")
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.is_stop, token.is_punct)

"""
Spacy has 5 attributes, all having to do with preprocessing (stopwords, lemmetized, POS, etc.)
"""

Today today NOUN False False
we we PRON True False
are be AUX True False
having have VERB False False
heavy heavy ADJ False False
rainfall rainfall NOUN False False
, , PUNCT False True
We we PRON True False
recommend recommend VERB False False
you you PRON True False
to to PART True False
stay stay VERB False False
at at ADP True False
your your PRON True False
home home NOUN False False
and and CCONJ True False
be be AUX True False
safe safe ADJ False False
, , PUNCT False True
Do do AUX True False
not not PART True False
start start VERB False False
running run VERB False False
here here ADV True False
and and CCONJ True False
there there ADV True False


In [54]:
def dataCleaning(sentence):
    doc = nlp(sentence)  # process the sentence with spacy: tokenizes, lemmatizes, tags POS, etc. all at once

    clean_tokens = []
    for token in doc:  # second pass: filter out unwanted tokens
        temp = token.lemma_.lower().strip()  # lemmatize + lowercase + strip whitespace
        if temp not in punct and temp not in stopwords:
            # keep the token only if it's NOT punctuation AND NOT a stopword
            clean_tokens.append(temp)
        else:
            pass

    return clean_tokens  # final list of cleaned, lemmatized, lowercased content words

## Here after passing a particular sentence in dataCleaning method we are returned with relevant words which contribute to the sentiments

In [55]:
dataCleaning("Today we are having heavy rainfall, We recommend you to stay at your home and be safe, Do not start running here and there")
# All the useful words are returned, no punctuations no stop words and in the lemmatized, tokenized form

['today',
 'heavy',
 'rainfall',
 'recommend',
 'stay',
 'home',
 'safe',
 'start',
 'run']

In [111]:
#Very good at lemmetizing and tokenization and everything uptop

In [57]:
# Spillting the train and test data
X = data['text']
y = data['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)
print(X_train.shape,y_test.shape)

(21984,) (5497,)


## Preparing Model

In [69]:
# Creating the model and pipeline
tfidf = TfidfVectorizer(tokenizer = dataCleaning) #use spacy as datacleaning and tokenizer
Lr = LogisticRegression() #support vector
steps = [('tfidf',tfidf),('Lr',Lr)]
pipe = Pipeline(steps)

In [70]:
# Training the model
pipe.fit(X_train,y_train) #uses tfidf, data = X_train and y_train, and once tfidf uses svm. <=== explanation of pipeline

C:\Users\rkim8\anaconda3\envs\yolo_project\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('Lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function dat...002B07063DC60>
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None


In [72]:
# Testing on the test dataset
y_pred = pipe.predict(X_test)

In [107]:
# Printing the classification report and the confusion matrix
print(classification_report(y_test,y_pred))
print("\n\n")
print(confusion_matrix(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.72      0.61      0.66      1560
           1       0.76      0.69      0.72      1699
           2       0.63      0.74      0.68      2238

    accuracy                           0.69      5497
   macro avg       0.70      0.68      0.69      5497
weighted avg       0.70      0.69      0.69      5497




[[ 947   88  525]
 [  75 1170  454]
 [ 287  288 1663]]


## Testing on the Random Manual Examples

**Here '1' represent that the input is positive sentiment**

In [74]:
# Testing on random inputs
pipe.predict(["Wow you are an amazing person"])

array([1])

In [109]:
pipe.predict(['Today we are having heavy rainfall, We recommend you to stay at your home and be safe, Do not start running here and there'])

array([2])

**Here '0' represent that input is negative sentiment**

**Here '1' represent that input is positive sentiment**

**Here '2' represent that input is negative sentiment**

In [95]:
pipe.predict(["Im very very very mid"])

array([2])

In [104]:
pipe.predict([""])

array([2])

### Footnotes
https://towardsdatascience.com/natural-language-processing-feature-engineering-using-tf-idf-e8b9d00e7e76

https://towardsdatascience.com/a-simple-example-of-pipeline-in-machine-learning-with-scikit-learn-e726ffbb6976


In [78]:
y_test

6605     2
11990    2
15184    2
24299    0
8574     2
        ..
9053     2
1085     1
27300    2
14147    2
20260    2
Name: sentiment, Length: 5497, dtype: int64

# Results

From human reinforcement from me (RLHF), it works very well. Logistic regression seems to work very well with the datacleansing/processing from Spacy. Thus, I can conclude that Spacy is extremely good at data preprocessing for sentiment analysis. Fast way to preprocess rather than manually doing everything.